In [17]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [18]:
GEN_1_GENETIC = CF_OUTPUTS / "gen1_models_explainers_constraints" / "genetic_exp"
GEN_1_GENETIC.is_dir()

True

In [19]:
batch_data = GEN_1_GENETIC / "XGB_prange_highthres_2026-04-17" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [20]:
pd.set_option("display.max_rows", None)

In [21]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [22]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [23]:
# correct_cf_idx
# Fix cf_id indexing - group by query_index and number counterfactuals sequentially
def fix_cf_id(df):
    # For each query_index group
    fixed_rows = []
    for query_idx, group in df.groupby('query_index', sort=False):
        group = group.copy()
        # First row is xin (original)
        xin_mask = group['cf_id'] == 'xin'
        cf_mask = ~xin_mask

        # Keep xin rows as is
        xin_rows = group[xin_mask].copy()

        # Fix cf rows
        cf_rows = group[cf_mask].copy()
        if len(cf_rows) > 0:
            # Number them cf_1, cf_2, cf_3, etc.
            cf_rows['cf_id'] = [f'cf_{i+1}' for i in range(len(cf_rows))]

        # Combine back
        fixed_rows.append(xin_rows)
        if len(cf_rows) > 0:
            fixed_rows.append(cf_rows)

    return pd.concat(fixed_rows, ignore_index=True)

In [24]:
batch_df = fix_cf_id(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.95,,,,0.026,
1,0,cf_1,,,,,,,17.5842,,,0.0722,1,True,0.026,0.003
2,0,cf_2,,,,7,,,18.9745,,,0.1256,2,False,0.026,0.0707
3,0,cf_3,,,,7,,,18.1954,,,0.1657,2,False,0.026,0.0222
4,0,cf_4,,,,,,,17.3749,5,,0.1722,2,True,0.026,0.0011
5,0,cf_5,,,,,,,17.2765,6,,0.1952,2,True,0.026,0.0126
6,0,cf_6,,,,6,,,16.7814,,,0.1968,2,False,0.026,0.0347
7,0,cf_7,,,6,,,,16.5582,,,0.25,2,True,0.026,0.005
8,0,cf_8,,,6,7,,,18.9745,,,0.2506,3,False,0.026,0.1428
9,0,cf_9,,,,7,,,18.9745,7,,0.2506,3,True,0.026,0.0038


In [25]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_cf_df = select_one_cf_per_query(batch_df)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,0.95,,,,0.026,
9,0,cf_1,,,,,,,17.5842,,,0.0722,1,True,0.026,0.003
1,1,xin,3,4,1,2,3,0,22.3757,0,0.95,,,,0.2497,
10,1,cf_6,,,,,1,,22.3757,1,,0.1475,3,True,0.2497,0.0691
2,2,xin,5,3,1,1,4,0,22.694,7,0.96,,,,0.218,
11,2,cf_3,,,,3,2,,22.6757,,,0.25,3,True,0.218,0.0426
3,3,xin,3,3,6,6,2,0,24.3418,1,1.04,,,,0.0653,
12,3,cf_1,,,,,,,24.3375,,,0.0001,1,False,0.0653,0.0653
4,4,xin,5,4,2,7,1,0,21.2585,3,1.14,,,,0.0811,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,True,0.0811,0.0141
